# Episode 5: Team of Agents: Taking Turns

What if you could put three experts in a room and have them take turns thinking through a problem?

**In this episode, you'll build:** A 3-agent debate team using round-robin turns.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen import ConversableAgent, LLMConfig

## From 2 agents to 3+

In Episode 4, two agents improved each other's work through back-and-forth review. But real teamwork often needs more than two perspectives — a proposer, a critic, and someone to synthesize the best of both.

Until now, you used `agent.run()` to start a conversation between two agents. For three or more, AG2 uses `initiate_group_chat()` with a **pattern** that controls who speaks next. The API looks different, but the concept is the same: agents take turns, each contributing based on their role.

The simplest pattern is **RoundRobinPattern** — each agent speaks in a fixed, repeating order, like taking turns around a conference table. This works well when every agent has something to contribute on every turn and you want predictable, structured conversations.

## Step 1: Create three debate agents

We'll set up a classic triad: someone to propose ideas, someone to stress-test them, and someone to find the synthesis. Let's see this in action:

In [ ]:
from dotenv import load_dotenv

load_dotenv()


llm_config = LLMConfig({"model": "gpt-4o-mini"})

proposer = ConversableAgent(
    name="proposer",
    system_message=(
        "You are a tech strategist who PROPOSES architectural decisions. "
        "Present clear arguments with concrete benefits. Keep responses to 2-3 paragraphs."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

challenger = ConversableAgent(
    name="challenger",
    system_message=(
        "You are a devil's advocate who CHALLENGES proposals. "
        "Find weaknesses, risks, and overlooked trade-offs. Be constructive but thorough. "
        "Keep responses to 2-3 paragraphs."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

refiner = ConversableAgent(
    name="refiner",
    system_message=(
        "You are a solution architect who REFINES ideas. "
        "Synthesize the proposer's ideas and the challenger's concerns into improved solutions. "
        "Suggest concrete compromises. Keep responses to 2-3 paragraphs."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

## Step 2: Set up RoundRobinPattern

The pattern definition is straightforward — you specify which agent goes first and the full list of participants. The framework handles the rotation.

In [ ]:
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import RoundRobinPattern

user = ConversableAgent(name="user", llm_config=llm_config, human_input_mode="NEVER")

pattern = RoundRobinPattern(
    initial_agent=proposer,
    agents=[proposer, challenger, refiner],
)

## Step 3: Run the debate

Six rounds means each agent speaks exactly twice. Watch the quality of the discussion evolve with each cycle.

In [ ]:
result, context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Should we use microservices or monolith for our new project?",
    max_rounds=6,
)

## What just happened?

The agents spoke in a fixed order: **proposer -> challenger -> refiner -> proposer -> challenger -> refiner**.

Each agent stayed in its role throughout. The proposer made the case, the challenger stress-tested it, and the refiner found the middle ground. Then the cycle repeated — and the second round was sharper than the first because each agent had more context to work with.

This propose-challenge-refine pattern mirrors how good teams actually debate decisions. The structure forces constructive friction.

## Trade-offs to consider

Round-robin is simple and predictable, which is both its strength and its limitation.

**It shines when** every voice matters on every turn — brainstorming sessions, structured debates, multi-perspective analysis, compliance reviews where legal, security, and architecture each need to weigh in. The fixed order means you can reason about the conversation flow before it happens.

**A different pattern is better when** you need dynamic routing (some agents may be irrelevant to certain turns), when the next speaker should depend on what was just said, or when you're watching costs closely — every agent speaks every round, even if they have nothing new to add.

## Mini-challenge

What happens if criticism comes *before* the proposal? Try changing the agent order to `[challenger, proposer, refiner]` so the challenger speaks first. How does starting with "what could go wrong" instead of "here's my idea" change the shape of the debate?

## Try it yourself

A few experiments to deepen your understanding:

1. **Add a 4th agent** — a `mediator` who summarizes points of agreement. Add it to the agents list and see how the dynamic shifts.
2. **Modify `max_rounds`** — try 3 (one round) vs 12 (four rounds). At what point does the debate stop improving?
3. **Change the topic** — try a non-technical topic like "Should our team work remotely or in-office?" and watch how the same structure produces very different conversations.

## Additional Content

### Alternative turn-taking patterns

AG2 offers other simple patterns besides RoundRobin, each suited to different situations.

**RandomPattern** shuffles the speaker order each round. This is useful for avoiding positional bias in debates — whoever goes first tends to anchor the conversation.

```python
from autogen.agentchat.group.patterns import RandomPattern

pattern = RandomPattern(
    initial_agent=proposer,
    agents=[proposer, challenger, refiner],
)
```

**ManualPattern** puts you in the driver's seat — you choose who speaks next at each step. This is great for interactive debugging and exploration.

```python
from autogen.agentchat.group.patterns import ManualPattern

pattern = ManualPattern(
    initial_agent=proposer,
    agents=[proposer, challenger, refiner],
)
```

## What's Next

Fixed turns are predictable, but what if the conversation itself could decide who should speak next? What if an LLM could read the room and pick the right expert?

That's exactly what we'll build in the next episode — same agents, but with smart speaker selection.

**Bonus:** Try the round-robin group chat demo in the [AG2 Playground](https://playground.ag2.ai).